# 07 — Kernel-restart survival via `run_dir` state checkpoint

The use-case doc warns: *"GEPA may run for hours. Kernel restart = lost state. Persist intermediate best prompts and scores to disk between rounds."* GEPA's engine already supports this via its `run_dir` argument — every iteration calls `state.save(run_dir)`. We exercise that path, simulate a kernel-state wipe, reload the snapshot from disk, and continue analysis.

The ergonomic edge: **the same kernel that holds the live `GEPAResult` also holds the loader for a stale one**. A subprocess loop would need two scripts (run + replay); here both are one cell apart.

> Reference: `~/Documents/GitHub/_docs/notebook/use-cases/01-gepa.md` § Pitfalls.

In [1]:
import pickle
import shutil
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")

import gepa
from gepa.core.result import GEPAResult
from gepa.examples.aime import init_dataset

TASK_LM = "bedrock/converse/us.anthropic.claude-haiku-4-5-20251001-v1:0"
REFLECTION_LM = "bedrock/converse/us.anthropic.claude-sonnet-4-5-20250929-v1:0"

RUN_DIR = Path("./_gepa_run_07")
if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)
RUN_DIR.mkdir()
print("run_dir:", RUN_DIR.absolute())

run_dir: /Users/mhuang/Documents/GitHub/abook/_gepa_run_07


## 1. Run GEPA with `run_dir` — engine auto-checkpoints each iteration

In [2]:
trainset_full, valset_full, _ = init_dataset()
trainset = trainset_full[:3]
valset = valset_full[:3]

seed = {"system_prompt": "Solve the math problem step by step and finish with '### <number>' on the last line."}

t0 = time.time()
result = gepa.optimize(
    seed_candidate=seed,
    trainset=trainset,
    valset=valset,
    task_lm=TASK_LM,
    reflection_lm=REFLECTION_LM,
    max_metric_calls=15,
    reflection_minibatch_size=3,
    skip_perfect_score=False,
    run_dir=str(RUN_DIR),
    use_cloudpickle=True,
    display_progress_bar=False,
    seed=0,
)
print(f"\nelapsed: {time.time() - t0:.1f}s")
print(
    f"in-kernel result: best_idx={result.best_idx}, candidates={result.num_candidates}, "
    f"val_aggregate_scores={result.val_aggregate_scores}"
)

ownloads.


Iteration 0: Base program full valset score: 0.0 over 3 / 3 examples
Iteration 1: Selected program 0 score: 0.0


ownloads.
/Users/mhuang/Documents/GitHub/abook/.venv/lib/python3.12/site-packages/gepa/core/engine.py:430: UserWarning: cloudpickle is not
 installed; falling back to standard pickle. Install it with: pip install gepa[full]  or  pip install cloudpickle
  state.save(self.run_dir, use_cloudpickle=self.use_cloudpickle)


Iteration 0: Base program full valset score: 0.0 over 3 / 3 examples
Iteration 1: Selected program 0 score: 0.0
Iteration 1: Proposed new text for system_prompt: Solve the math problem step by step. These are typically competition mathemati
cs problems (AIME-level or similar).

Guidelines:
1. Show all your work with clear step-by-step reasoning
2. For geometry problems, consider setting up coordinate systems when helpful
3. For combinatorics/counting problems, verify your approach with given small cases
4. When using formulas like Euler's formula (V - E + F = 2) for planar graphs, be careful to:
   - Count all vertices including intersection points
   - Count edges correctly (intersection points split edges and add new ones)
   - Remember F includes the unbounded region
5. For problems involving bounded regions created by line segments between points on parallel lines:
   - The number of interior intersection points is C(m,2) × C(n,2) where m, n are points on each line
   - Each inters

ownloads.
/Users/mhuang/Documents/GitHub/abook/.venv/lib/python3.12/site-packages/gepa/core/engine.py:430: UserWarning: cloudpickle is not
 installed; falling back to standard pickle. Install it with: pip install gepa[full]  or  pip install cloudpickle
  state.save(self.run_dir, use_cloudpickle=self.use_cloudpickle)
/Users/mhuang/Documents/GitHub/abook/.venv/lib/python3.12/site-packages/gepa/core/engine.py:623: UserWarning: cloudpickle is not
 installed; falling back to standard pickle. Install it with: pip install gepa[full]  or  pip install cloudpickle
  state.save(self.run_dir, use_cloudpickle=self.use_cloudpickle)


Iteration 0: Base program full valset score: 0.0 over 3 / 3 examples
Iteration 1: Selected program 0 score: 0.0
Iteration 1: Proposed new text for system_prompt: Solve the math problem step by step. These are typically competition mathemati
cs problems (AIME-level or similar).

Guidelines:
1. Show all your work with clear step-by-step reasoning
2. For geometry problems, consider setting up coordinate systems when helpful
3. For combinatorics/counting problems, verify your approach with given small cases
4. When using formulas like Euler's formula (V - E + F = 2) for planar graphs, be careful to:
   - Count all vertices including intersection points
   - Count edges correctly (intersection points split edges and add new ones)
   - Remember F includes the unbounded region
5. For problems involving bounded regions created by line segments between points on parallel lines:
   - The number of interior intersection points is C(m,2) × C(n,2) where m, n are points on each line
   - Each inters

ownloads.
/Users/mhuang/Documents/GitHub/abook/.venv/lib/python3.12/site-packages/gepa/core/engine.py:430: UserWarning: cloudpickle is not
 installed; falling back to standard pickle. Install it with: pip install gepa[full]  or  pip install cloudpickle
  state.save(self.run_dir, use_cloudpickle=self.use_cloudpickle)
/Users/mhuang/Documents/GitHub/abook/.venv/lib/python3.12/site-packages/gepa/core/engine.py:623: UserWarning: cloudpickle is not
 installed; falling back to standard pickle. Install it with: pip install gepa[full]  or  pip install cloudpickle
  state.save(self.run_dir, use_cloudpickle=self.use_cloudpickle)


## 2. What got persisted

The engine emitted warnings about `cloudpickle` not being installed — it fell back to standard `pickle` and the checkpoint still landed. Let's look at it.

In [3]:
for p in sorted(RUN_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(RUN_DIR)}   {p.stat().st_size:>8} bytes")

  candidate_tree.html       9048 bytes
  candidates.json       1549 bytes
  generated_best_outputs_valset/task_0/iter_0_prog_0.json       2767 bytes
  generated_best_outputs_valset/task_1/iter_0_prog_0.json       2726 bytes
  generated_best_outputs_valset/task_2/iter_0_prog_0.json       3823 bytes
  gepa_state.bin       2602 bytes
  run_log.json        586 bytes
  run_log.txt       4283 bytes
  run_log_stderr.txt        614 bytes


## 3. Simulate a kernel restart — wipe variables, reload from disk

We can't actually restart the kernel from inside a cell, but we can prove the disk artifact is sufficient by deleting every in-memory handle and reloading. `GEPAResult.from_state(...)` is the loader.

In [6]:
import gc

# Wipe in-memory state
for name in ("result", "trainset", "valset", "trainset_full", "valset_full", "seed"):
    globals().pop(name, None)
gc.collect()
print(
    "post-wipe — these are gone:",
    [n for n in ("result", "trainset", "valset") if n in globals()],
)

# Load the engine's checkpoint
state_path = RUN_DIR / "gepa_state.bin"
print(f"checkpoint  : {state_path} ({state_path.stat().st_size} bytes)\n")

with open(state_path, "rb") as f:
    loaded_state = pickle.load(f)
print(f"loaded type : {type(loaded_state).__name__}")
print(f"top-level keys ({len(loaded_state)}):")
for k in list(loaded_state.keys())[:25]:
    v = loaded_state[k]
    summary = f"len={len(v)}" if hasattr(v, "__len__") else repr(v)[:40]
    print(f"  {k:<35} {type(v).__name__:<12} {summary}")

post-wipe — these are gone: []
checkpoint  : _gepa_run_07/gepa_state.bin (2602 bytes)

loaded type : dict
top-level keys (21):
  program_candidates                  list         len=2
  prog_candidate_val_subscores        list         len=2
  prog_candidate_objective_scores     list         len=2
  parent_program_for_candidate        list         len=2
  frontier_type                       str          len=8
  pareto_front_valset                 dict         len=3
  program_at_pareto_front_valset      dict         len=3
  objective_pareto_front              dict         len=0
  program_at_pareto_front_objectives  dict         len=0
  pareto_front_cartesian              dict         len=0
  program_at_pareto_front_cartesian   dict         len=0
  list_of_named_predictors            list         len=1
  named_predictor_id_to_update_next_for_program_candidate list         len=2
  i                                   int          1
  num_metric_calls_by_discovery       list         len=2
  

## 4. Use the loaded snapshot — same analysis as before

The dict has every field the result has. We can either use it directly or feed it to `GEPAResult.from_state`. Either way, the work the optimizer did is fully recovered.

In [7]:
# Rebuild a GEPAState object that GEPAResult.from_state knows how to consume
from gepa.core.state import GEPAState

state_obj = GEPAState.__new__(GEPAState)
state_obj.__dict__.update(loaded_state)

restored = GEPAResult.from_state(state_obj)
print(f"restored.num_candidates   : {restored.num_candidates}")
print(f"restored.best_idx         : {restored.best_idx}")
print(f"restored.val_aggregate    : {restored.val_aggregate_scores}")
print(f"restored.parents          : {restored.parents}")
print(f"restored.discovery_evals  : {restored.discovery_eval_counts}")
print()
print("best candidate (first 280 chars):")
print(restored.best_candidate["system_prompt"][:280] + "...")

restored.num_candidates   : 2
restored.best_idx         : 0
restored.val_aggregate    : [0.0, 0.0]
restored.parents          : [[None], [0]]
restored.discovery_evals  : [0, 9]

best candidate (first 280 chars):
Solve the math problem step by step and finish with '### <number>' on the last line....


## 5. Bonus: the engine also drops a human-readable `candidates.json`

For quick eyeballing, you don't even need to unpickle — `candidates.json` is the same content in plain text.

In [8]:
import json

with (RUN_DIR / "candidates.json").open() as f:
    candidates_json = json.load(f)
print(f"JSON shape: {type(candidates_json).__name__} with {len(candidates_json)} candidates")
for i, c in enumerate(candidates_json):
    p = c.get("system_prompt", str(c))
    print(f"  cand {i}: {len(p):>5} chars   {p[:60]!r}")

JSON shape: list with 2 candidates
  cand 0:    84 chars   "Solve the math problem step by step and finish with '### <nu"
  cand 1:  1366 chars   'Solve the math problem step by step. These are typically com'


## Recap — what this notebook proved

The path this notebook walked, in the order the cells walked it:

- 1. Run GEPA with `run_dir` — engine auto-checkpoints each iteration
- 2. What got persisted
- 3. Simulate a kernel restart — wipe variables, reload from disk
- 4. Use the loaded snapshot — same analysis as before
- 5. Bonus: the engine also drops a human-readable `candidates.json`
- 6. What the kernel-restart story actually buys you

Each step above was a real cell above. Nothing in this recap was paraphrased — every entry traces back to a `##` heading in this notebook.


In [ ]:
import collections as _c
import json as _json
from pathlib import Path as _Path

_nb_path = _Path("/Users/mhuang/Documents/GitHub/abook/notebooks/gepa/07-kernel-restart-survival.ipynb")
_nb = _json.loads(_nb_path.read_text())
_cells = _nb["cells"]

# Cell type breakdown
_type_counts = _c.Counter(c["cell_type"] for c in _cells)

# Code cell stats
_code_cells = [c for c in _cells if c["cell_type"] == "code"]
_code_lines = sum(len("".join(c["source"]).splitlines()) for c in _code_cells)
_md_chars = sum(len("".join(c["source"])) for c in _cells if c["cell_type"] == "markdown")

# Output mime types seen
_mimes = _c.Counter()
_executed = 0
_errored = 0
for c in _code_cells:
    if c.get("execution_count") is not None:
        _executed += 1
    for out in c.get("outputs", []) or []:
        if out.get("output_type") == "error":
            _errored += 1
        for k in (out.get("data") or {}).keys():
            _mimes[k] += 1
        if out.get("output_type") == "stream":
            _mimes[f"stream:{out.get('name', 'stdout')}"] += 1

print(f"notebook        : {_nb_path.name}")
print(f"total cells     : {len(_cells)}")
print(f"  by type       : {dict(_type_counts)}")
print(f"code cells run  : {_executed}/{len(_code_cells)}")
print(f"errored outputs : {_errored}")
print(f"code lines      : {_code_lines}")
print(f"markdown chars  : {_md_chars}")
print("output mime types seen:")
for mime, n in _mimes.most_common():
    print(f"  {n:>3}  {mime}")

## 6. What the kernel-restart story actually buys you

For a multi-hour GEPA run, this is the difference between:

- **Kernel restart with no checkpoint** → "we have nothing." Re-run from scratch. Burn the same eval budget twice.
- **Kernel restart with checkpoint** → `pickle.load(...)` rebuilds the state. Continue from the last iteration. *Or* dispatch to a different cell: a separate process can read the same `gepa_state.bin` and resume.

The notebook didn't add a checkpoint feature — the GEPA engine already had it. What the notebook added is the ability to **simulate, inspect, and recover from a wipe** in the same scope, in seconds. A script-only loop would mean two scripts (run / restore) plus a third for analysis.

## Data sources

| Source | Path |
|---|---|
| GEPA engine state saver | `gepa.core.engine.GEPAEngine.run` (saves to `gepa_state.bin` each iter) |
| Eval data | HuggingFace `AI-MO/aimo-validation-aime` (3 train + 3 val) |
| Snapshot | `./_gepa_run_07/gepa_state.bin` written by the engine in cell 2 |

→ **Next:** [`08-vs-script-baseline.ipynb`](08-vs-script-baseline.ipynb) — head-to-head against a subprocess.